# Quantization + Serving

Quantize a small Transformer to INT8/INT4/FP8/AWQ, measure SNR, estimate
size, and launch `sneppx-serve`. The quantization math runs on pure NumPy.

In [ ]:
import numpy as np
from SneppX_ALG import (
    Transformer, Tensor, QuantMode,
    quantize_int8_sym, dequantize_int8_sym, quantize_int8_channel, dequantize_int8_channel,
    quantize_int4_sym, dequantize_int4_sym, awq_quantize, quantize_error,
)
from SneppX_ALG.interface_bindings.quantized_serve import (
    QuantizedModelConfig, quantize_model_weights, dequantize_weights,
    estimate_model_size_mb,
)

model = Transformer(vocab_size=500, dim=128, num_heads=4, num_layers=2,
                    ffn_dim=256, max_seq_len=64)
params = {n: p.data.copy() for n, p in model.named_parameters()}
print('FP32 MB:', round(sum(v.nbytes for v in params.values())/1e6, 2))

## Per-tensor INT8

In [ ]:
w = Tensor.from_numpy(params['lm_head.weight'].astype(np.float32))
qw, scale = quantize_int8_sym(w)
wq = dequantize_int8_sym(qw, scale)
print('INT8 SNR (dB):', round(quantize_error(w, wq, metric='snr'), 2))

## Per-channel INT8 (lower error)

In [ ]:
qw, scales = quantize_int8_channel(w, dim=-1)
wq = dequantize_int8_channel(qw, scales)
print('per-channel SNR (dB):', round(quantize_error(w, wq, metric='snr'), 2))

## INT4 AWQ

In [ ]:
act_scales = Tensor.randn((w.shape[1],))
qw, scales = awq_quantize(w, act_scales, group_size=64)
print('AWQ INT4 quantized, scale shape:', scales.shape)

## Whole-model quantization + size estimate

In [ ]:
cfg = QuantizedModelConfig(quant_mode=QuantMode.INT4_SYM,
                          skip_layers=['lm_head', 'embedding.weight'])
quantized = quantize_model_weights(params, cfg)
print('INT4 MB:', round(estimate_model_size_mb(quantized), 2))

## Serve with sneppx-serve

In [ ]:
import json
weights = {k: (v.weight.data if hasattr(v, 'weight') else v)
            for k, v in quantized.items()}
np.savez('/tmp/q_model.npz', **weights)
json.dump({'quant_mode': QuantMode.INT4_SYM, 'skip_layers': ['lm_head']},
          open('/tmp/q_config.json', 'w'))
print('saved /tmp/q_model.npz + /tmp/q_config.json')
# Then from a shell:
#   sneppx-serve --port 8000 --api-keys dev-key \
#     --model-config /tmp/q_config.json --checkpoint /tmp/q_model.npz